In [9]:
pip install pandas numpy openpyxl sqlalchemy pyodbc prophet scikit-learn matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np

In [11]:
file_path = "Retail_Profit_Leakage_Dataset.xlsx"

sales = pd.read_excel(
    file_path,
    sheet_name='Sales_Transactions'
)

products = pd.read_excel(
    file_path,
    sheet_name='Product_Master'
)

inventory = pd.read_excel(
    file_path,
    sheet_name='Inventory_Daily'
)

returns = pd.read_excel(
    file_path,
    sheet_name='Returns'
)

vendor = pd.read_excel(
    file_path,
    sheet_name='Vendor_Supply'
)

promotions = pd.read_excel(
    file_path,
    sheet_name='Promotions'
)

customers = pd.read_excel(
    file_path,
    sheet_name='Customer_Master'
)

In [13]:
print(sales.head(20))



      Order_ID           Order_Date Store_ID Customer_ID    SKU_ID  Quantity  \
0   ORD1000000  2025-06-01 00:00:00   HYD-03  CUST007684  SKU00565         4   
1   ORD1000001  2025-03-14 00:00:00   HYD-03  CUST004865  SKU01082         6   
2   ORD1000002  2025-06-05 00:00:00   BEN-05  CUST006245  SKU00901         2   
3   ORD1000003  2025-06-23 00:00:00   JAI-02  CUST009196  SKU00524         2   
4   ORD1000004  2025-03-01 00:00:00   PUN-01  CUST000981  SKU00215         2   
5   ORD1000005  2025-05-02 00:00:00   AHM-02  CUST005962  SKU01171         2   
6   ORD1000006  2025-05-26 00:00:00   CHE-01  CUST010497  SKU00267         3   
7   ORD1000007  2025-06-01 00:00:00   CHE-01  CUST004609  SKU00418         4   
8   ORD1000008  2025-04-13 00:00:00   CHE-03  CUST005707  SKU00540         2   
9   ORD1000009  2025-06-16 00:00:00   BEN-03  CUST001059  SKU00396         2   
10  ORD1000010  2025-03-16 00:00:00   HYD-05  CUST011338  SKU01048         2   
11  ORD1000011  2025-04-23 00:00:00   MU

In [14]:
print(sales.isnull().sum())
print(products.isnull().sum())
print(inventory.isnull().sum())
print(returns.isnull().sum())
print(vendor.isnull().sum())
print(promotions.isnull().sum())
print(customers.isnull().sum())

Order_ID               0
Order_Date             0
Store_ID               0
Customer_ID            0
SKU_ID                 0
Quantity               0
Selling_Price        326
Discount_Amount      326
Promo_ID           51469
Delivery_Type          0
Payment_Mode         326
Order_Status           0
dtype: int64
SKU_ID           0
SKU_Name         0
Category         0
Sub_Category     0
Brand           12
Tax_Percent      0
Unit_Cost       12
MRP              0
dtype: int64
Date                 0
Store_ID             0
SKU_ID               0
Reorder_Level        0
Opening_Stock        0
Closing_Stock      252
Warehouse_Stock    252
dtype: int64
Return_ID         0
Order_ID          0
SKU_ID            0
Return_Date       0
Return_Reason    47
Refund_Amount     0
Return_Status     0
dtype: int64
PO_ID                       0
Vendor_ID                   0
SKU_ID                      0
Store_ID                    0
PO_Date                     0
Expected_Delivery_Date      0
Actual_Delivery

In [16]:
# =========================
# HANDLE NULL VALUES
# =========================

# ---------- SALES ----------
sales['Discount_Amount'] = sales['Discount_Amount'].fillna(0)

sales['Promo_ID'] = sales['Promo_ID'].fillna('No_Promo')

sales['Payment_Mode'] = sales['Payment_Mode'].fillna(
    sales['Payment_Mode'].mode()[0]
)

sales['Selling_Price'] = sales['Selling_Price'].fillna(
    sales['Selling_Price'].median()
)


# ---------- PRODUCTS ----------
products['Brand'] = products['Brand'].fillna('Unknown')

products['Unit_Cost'] = products.groupby('Category')['Unit_Cost'] \
    .transform(lambda x: x.fillna(x.median()))

# If still any null values remain
products['Unit_Cost'] = products['Unit_Cost'].fillna(
    products['Unit_Cost'].median()
)


# ---------- INVENTORY ----------
inventory['Closing_Stock'] = inventory['Closing_Stock'].fillna(
    inventory['Opening_Stock']
)

inventory['Warehouse_Stock'] = inventory['Warehouse_Stock'].fillna(
    inventory['Warehouse_Stock'].median()
)


# ---------- RETURNS ----------
returns['Return_Reason'] = returns['Return_Reason'].fillna(
    'Unknown'
)


# ---------- VENDOR ----------
vendor['Procurement_Cost'] = vendor['Procurement_Cost'].fillna(
    vendor['Procurement_Cost'].median()
)


# ---------- PROMOTIONS ----------
promotions['Promo_Budget'] = promotions['Promo_Budget'].fillna(
    promotions['Promo_Budget'].median()
)


# ---------- CUSTOMERS ----------
customers['City'] = customers['City'].fillna(
    'Unknown'
)


# =========================
# VERIFY NULL VALUES
# =========================

print("SALES NULLS")
print(sales.isnull().sum())

print("\nPRODUCTS NULLS")
print(products.isnull().sum())

print("\nINVENTORY NULLS")
print(inventory.isnull().sum())

print("\nRETURNS NULLS")
print(returns.isnull().sum())

print("\nVENDOR NULLS")
print(vendor.isnull().sum())

print("\nPROMOTIONS NULLS")
print(promotions.isnull().sum())

print("\nCUSTOMERS NULLS")
print(customers.isnull().sum())

SALES NULLS
Order_ID           0
Order_Date         0
Store_ID           0
Customer_ID        0
SKU_ID             0
Quantity           0
Selling_Price      0
Discount_Amount    0
Promo_ID           0
Delivery_Type      0
Payment_Mode       0
Order_Status       0
dtype: int64

PRODUCTS NULLS
SKU_ID          0
SKU_Name        0
Category        0
Sub_Category    0
Brand           0
Tax_Percent     0
Unit_Cost       0
MRP             0
dtype: int64

INVENTORY NULLS
Date               0
Store_ID           0
SKU_ID             0
Reorder_Level      0
Opening_Stock      0
Closing_Stock      0
Warehouse_Stock    0
dtype: int64

RETURNS NULLS
Return_ID        0
Order_ID         0
SKU_ID           0
Return_Date      0
Return_Reason    0
Refund_Amount    0
Return_Status    0
dtype: int64

VENDOR NULLS
PO_ID                     0
Vendor_ID                 0
SKU_ID                    0
Store_ID                  0
PO_Date                   0
Expected_Delivery_Date    0
Actual_Delivery_Date      0
Su

In [17]:
sales.drop_duplicates(inplace=True)
products.drop_duplicates(inplace=True)
inventory.drop_duplicates(inplace=True)
returns.drop_duplicates(inplace=True)
vendor.drop_duplicates(inplace=True)
promotions.drop_duplicates(inplace=True)
customers.drop_duplicates(inplace=True)

In [18]:
sales = sales[sales['Quantity'] > 0]

In [19]:
sales = sales[
    sales['Order_Status'] != 'Cancelled'
]

In [20]:
sales['Store_ID'] = (
    sales['Store_ID']
    .str.upper()
    .str.replace('-', '')
    .str.replace('_', '')
)

In [21]:
missing_sku = sales[
    ~sales['SKU_ID'].isin(products['SKU_ID'])
]

print(missing_sku)

         Order_ID           Order_Date Store_ID Customer_ID     SKU_ID  \
47     ORD1000047           32/13/2025    KOC02  CUST007110   SKU00178   
93     ORD1000093  2025-05-09 00:00:00    KOL04  CUST004921  SKU-39322   
110    ORD1000110  2025-06-01 00:00:00    MUM01  CUST006912  SKU-95439   
120    ORD1000120  2025-02-05 00:00:00    MUM05  CUST011557  SKU-54918   
126    ORD1000126  2025-01-21 00:00:00    HYD05  CUST007074  SKU-54114   
...           ...                  ...      ...         ...        ...   
81442  ORD1033275  2025-03-06 00:00:00    CHE01  CUST003042  SKU-80134   
81461  ORD1042075  2025-01-05 00:00:00    CHE03  CUST003004  SKU-92028   
81525  ORD1001570  2025-05-06 00:00:00    DEL05  CUST007301  SKU-52946   
81569  ORD1043398  2025-04-23 00:00:00    AHM04  CUST009937  SKU-91368   
81580  ORD1044331  2025-05-03 00:00:00    BEN05  CUST009277  SKU-19228   

       Quantity  Selling_Price  Discount_Amount    Promo_ID  Delivery_Type  \
47            3          63.27   

In [22]:
invalid_promos = sales[
    ~sales['Promo_ID']
    .isin(promotions['Promo_ID'])
]

print(invalid_promos)

         Order_ID           Order_Date Store_ID Customer_ID     SKU_ID  \
0      ORD1000000  2025-06-01 00:00:00    HYD03  CUST007684   SKU00565   
1      ORD1000001  2025-03-14 00:00:00    HYD03  CUST004865   SKU01082   
2      ORD1000002  2025-06-05 00:00:00    BEN05  CUST006245   SKU00901   
4      ORD1000004  2025-03-01 00:00:00    PUN01  CUST000981   SKU00215   
5      ORD1000005  2025-05-02 00:00:00    AHM02  CUST005962   SKU01171   
...           ...                  ...      ...         ...        ...   
81563  ORD1050773  2025-04-18 00:00:00    PUN02  CUST009424   SKU00883   
81565  ORD1037857  2025-04-11 00:00:00    KOL02  CUST004828   SKU01060   
81569  ORD1043398  2025-04-23 00:00:00    AHM04  CUST009937  SKU-91368   
81574  ORD1061636  2025-03-03 00:00:00    PUN4   CUST008513   SKU00498   
81580  ORD1044331  2025-05-03 00:00:00    BEN05  CUST009277  SKU-19228   

       Quantity  Selling_Price  Discount_Amount    Promo_ID  Delivery_Type  \
0             4          72.73   

In [23]:
invalid_returns = returns[
    ~returns['Order_ID']
    .isin(sales['Order_ID'])
]

print(invalid_returns)

        Return_ID     Order_ID    SKU_ID          Return_Date  Return_Reason  \
99      RET200099   ORD1010400  SKU00855  2025-05-21 00:00:00        Damaged   
562     RET200562   ORD1047923  SKU01194  2025-04-06 00:00:00     Wrong Item   
987     RET200987   ORD1026696  SKU01165  2025-04-01 00:00:00        Damaged   
1178    RET201178   ORD1068088  SKU00204  2025-04-14 00:00:00   Changed Mind   
1309    RET201309   ORD1029653  SKU00501  2025-01-17 00:00:00   Changed Mind   
...           ...          ...       ...                  ...            ...   
4696  RET-1794775  ORD-2921470  SKU00191  2025-03-24 00:00:00  Late Delivery   
4697  RET-2924226  ORD-4438000  SKU00195  2025-03-26 00:00:00        Damaged   
4698  RET-8454564  ORD-3418454  SKU00447  2025-04-05 00:00:00        Damaged   
4699  RET-6793261  ORD-8923464  SKU00699  2025-07-08 00:00:00     Wrong Item   
4700  RET-2451482  ORD-8661457  SKU00546  2025-03-05 00:00:00     Wrong Item   

      Refund_Amount Return_Status  
99 

In [24]:
# Convert dates safely and keep dd-mm-yyyy

customers['Signup_Date'] = pd.to_datetime(
    customers['Signup_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

inventory['Date'] = pd.to_datetime(
    inventory['Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

promotions['Start_Date'] = pd.to_datetime(
    promotions['Start_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

promotions['End_Date'] = pd.to_datetime(
    promotions['End_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

returns['Return_Date'] = pd.to_datetime(
    returns['Return_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

sales['Order_Date'] = pd.to_datetime(
    sales['Order_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

vendor['PO_Date'] = pd.to_datetime(
    vendor['PO_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

vendor['Expected_Delivery_Date'] = pd.to_datetime(
    vendor['Expected_Delivery_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

vendor['Actual_Delivery_Date'] = pd.to_datetime(
    vendor['Actual_Delivery_Date'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%d-%m-%Y')

In [25]:
# Fill missing dates for all tables

customers['Signup_Date'] = customers['Signup_Date'].ffill().bfill()

inventory['Date'] = inventory['Date'].ffill().bfill()

promotions['Start_Date'] = promotions['Start_Date'].ffill().bfill()
promotions['End_Date'] = promotions['End_Date'].ffill().bfill()

returns['Return_Date'] = returns['Return_Date'].ffill().bfill()

sales['Order_Date'] = sales['Order_Date'].ffill().bfill()

vendor['PO_Date'] = vendor['PO_Date'].ffill().bfill()
vendor['Expected_Delivery_Date'] = vendor['Expected_Delivery_Date'].ffill().bfill()
vendor['Actual_Delivery_Date'] = vendor['Actual_Delivery_Date'].ffill().bfill()

In [26]:
print("Customers:", customers['Signup_Date'].isna().sum())

print("Inventory:", inventory['Date'].isna().sum())

print("Promotions Start:", promotions['Start_Date'].isna().sum())
print("Promotions End:", promotions['End_Date'].isna().sum())

print("Returns:", returns['Return_Date'].isna().sum())

print("Sales:", sales['Order_Date'].isna().sum())

print("Vendor PO:", vendor['PO_Date'].isna().sum())
print("Vendor Expected:", vendor['Expected_Delivery_Date'].isna().sum())
print("Vendor Actual:", vendor['Actual_Delivery_Date'].isna().sum())

Customers: 0
Inventory: 0
Promotions Start: 0
Promotions End: 0
Returns: 0
Sales: 0
Vendor PO: 0
Vendor Expected: 0
Vendor Actual: 0


In [27]:
# =========================
# EXPORT CLEANED TABLES
# =========================

sales.to_excel(
    'Cleaned_Sales.xlsx',
    index=False
)

products.to_excel(
    'Cleaned_Products.xlsx',
    index=False
)

inventory.to_excel(
    'Cleaned_Inventory.xlsx',
    index=False
)

returns.to_excel(
    'Cleaned_Returns.xlsx',
    index=False
)

vendor.to_excel(
    'Cleaned_Vendor.xlsx',
    index=False
)

promotions.to_excel(
    'Cleaned_Promotions.xlsx',
    index=False
)

customers.to_excel(
    'Cleaned_Customers.xlsx',
    index=False
)

print("All cleaned tables exported successfully.")

All cleaned tables exported successfully.


In [28]:
# =========================================================
# ADVANCED RETAIL METRICS - FINAL CLEAN CODE
# =========================================================

import pandas as pd
import numpy as np


# =========================================================
# 1. GROSS MARGIN %
# =========================================================

sales_margin = sales.merge(
    products[['SKU_ID', 'Unit_Cost']],
    on='SKU_ID',
    how='left'
)

sales_margin['Revenue'] = (
    sales_margin['Selling_Price'] *
    sales_margin['Quantity']
)

sales_margin['Cost'] = (
    sales_margin['Unit_Cost'] *
    sales_margin['Quantity']
)

sales_margin['Gross_Profit'] = (
    sales_margin['Revenue'] -
    sales_margin['Cost']
)

sales_margin['Gross_Margin_%'] = (
    sales_margin['Gross_Profit'] /
    sales_margin['Revenue']
) * 100

print("Gross Margin Created")


# =========================================================
# 2. NET MARGIN
# =========================================================

sales_margin['Net_Revenue'] = (
    sales_margin['Revenue'] -
    sales_margin['Discount_Amount']
)

refunds = returns.groupby(
    'Order_ID'
)['Refund_Amount'].sum().reset_index()

sales_margin = sales_margin.merge(
    refunds,
    on='Order_ID',
    how='left'
)

sales_margin['Refund_Amount'] = (
    sales_margin['Refund_Amount']
    .fillna(0)
)

sales_margin['Net_Profit'] = (
    sales_margin['Net_Revenue'] -
    sales_margin['Refund_Amount'] -
    sales_margin['Cost']
)

sales_margin['Net_Margin_%'] = (
    sales_margin['Net_Profit'] /
    sales_margin['Net_Revenue']
) * 100

print("Net Margin Created")


# =========================================================
# 3. STOCKOUT LOSS ESTIMATE
# =========================================================

demand = (
    sales.groupby('SKU_ID')['Quantity']
    .mean()
    .reset_index()
)

demand.rename(
    columns={'Quantity': 'Avg_Demand'},
    inplace=True
)

stockout = inventory.merge(
    demand,
    on='SKU_ID',
    how='left'
)

stockout['Avg_Demand'] = (
    stockout['Avg_Demand']
    .fillna(0)
)

stockout['Stockout'] = np.where(
    (
        stockout['Closing_Stock'] == 0
    ) &
    (
        stockout['Avg_Demand'] > 0
    ),
    'Yes',
    'No'
)

stockout['Potential_Lost_Sales'] = np.where(
    stockout['Stockout'] == 'Yes',
    stockout['Avg_Demand'],
    0
)

print("Stockout Metric Created")


# =========================================================
# 4. FILL RATE
# =========================================================

vendor['Fill_Rate_%'] = 100

print("Fill Rate Created")


# =========================================================
# 5. PROMO ROI
# =========================================================

promo_sales = sales.merge(
    promotions,
    on='Promo_ID',
    how='left'
)

promo_sales['Revenue'] = (
    promo_sales['Selling_Price'] *
    promo_sales['Quantity']
)

promo_roi = (
    promo_sales.groupby('Promo_ID')
    .agg({
        'Revenue': 'sum',
        'Discount_Amount': 'sum',
        'Promo_Budget': 'first'
    })
    .reset_index()
)

promo_roi['Incremental_Profit'] = (
    promo_roi['Revenue'] -
    promo_roi['Discount_Amount']
)

promo_roi['Promo_ROI'] = (
    promo_roi['Incremental_Profit'] /
    promo_roi['Promo_Budget']
)

print("Promo ROI Created")


# =========================================================
# 6. RETURN RATE BY CATEGORY
# =========================================================

return_data = returns.merge(
    products[['SKU_ID', 'Category']],
    on='SKU_ID',
    how='left'
)

return_rate = (
    return_data.groupby('Category')
    ['Return_ID']
    .count()
    .reset_index()
)

return_rate.rename(
    columns={
        'Return_ID': 'Total_Returns'
    },
    inplace=True
)

sales_count = sales.merge(
    products[['SKU_ID', 'Category']],
    on='SKU_ID',
    how='left'
)

sales_count = (
    sales_count.groupby('Category')
    ['Order_ID']
    .count()
    .reset_index()
)

sales_count.rename(
    columns={
        'Order_ID': 'Total_Orders'
    },
    inplace=True
)

return_rate = return_rate.merge(
    sales_count,
    on='Category',
    how='left'
)

return_rate['Return_Rate_%'] = (
    return_rate['Total_Returns'] /
    return_rate['Total_Orders']
) * 100

print("Return Rate Created")


# =========================================================
# 7. REPEAT PURCHASE RATE
# =========================================================

customer_orders = (
    sales.groupby('Customer_ID')
    ['Order_ID']
    .nunique()
    .reset_index()
)

repeat_customers = customer_orders[
    customer_orders['Order_ID'] > 1
]

repeat_rate = (
    len(repeat_customers) /
    len(customer_orders)
) * 100

print("\nRepeat Purchase Rate:")
print(round(repeat_rate, 2))


# =========================================================
# 8. VENDOR SLA BREACH %
# (WARNING FIXED)
# =========================================================

vendor['Expected_Delivery_Date'] = pd.to_datetime(
    vendor['Expected_Delivery_Date'],
    format='%d-%m-%Y',
    errors='coerce'
)

vendor['Actual_Delivery_Date'] = pd.to_datetime(
    vendor['Actual_Delivery_Date'],
    format='%d-%m-%Y',
    errors='coerce'
)

vendor['SLA_Breach'] = np.where(
    vendor['Actual_Delivery_Date'] >
    vendor['Expected_Delivery_Date'],
    'Late',
    'On Time'
)

sla_percent = (
    (vendor['SLA_Breach'] == 'Late')
    .mean()
) * 100

print("\nVendor SLA Breach %:")
print(round(sla_percent, 2))


# =========================================================
# CHECK INVALID DATES
# =========================================================

bad_dates = vendor[
    vendor['Expected_Delivery_Date'].isna() |
    vendor['Actual_Delivery_Date'].isna()
]

print("\nInvalid Date Records:")
print(bad_dates)


# Convert back for export
vendor['Expected_Delivery_Date'] = (
    vendor['Expected_Delivery_Date']
    .dt.strftime('%d-%m-%Y')
)

vendor['Actual_Delivery_Date'] = (
    vendor['Actual_Delivery_Date']
    .dt.strftime('%d-%m-%Y')
)


# =========================================================
# EXPORT REPORTS
# =========================================================

sales_margin.to_excel(
    'Sales_Margin_Report.xlsx',
    index=False
)

stockout.to_excel(
    'Stockout_Report.xlsx',
    index=False
)

promo_roi.to_excel(
    'Promo_ROI_Report.xlsx',
    index=False
)

return_rate.to_excel(
    'Return_Rate_Report.xlsx',
    index=False
)

vendor.to_excel(
    'Vendor_SLA_Report.xlsx',
    index=False
)

print("\nAll reports exported successfully.")

Gross Margin Created
Net Margin Created
Stockout Metric Created
Fill Rate Created
Promo ROI Created
Return Rate Created

Repeat Purchase Rate:
96.36

Vendor SLA Breach %:
30.41

Invalid Date Records:
Empty DataFrame
Columns: [PO_ID, Vendor_ID, SKU_ID, Store_ID, PO_Date, Expected_Delivery_Date, Actual_Delivery_Date, Supplied_Quantity, Procurement_Cost, Fill_Rate_%, SLA_Breach]
Index: []

All reports exported successfully.


In [29]:
# =========================================================
# CREATE DATA QUALITY REPORT
# =========================================================

report = []

tables = {
    'Sales': sales,
    'Products': products,
    'Inventory': inventory,
    'Returns': returns,
    'Vendor': vendor,
    'Promotions': promotions,
    'Customers': customers
}

for name, df in tables.items():

    report.append({
        'Table_Name': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Null_Values': df.isnull().sum().sum(),
        'Duplicate_Rows': df.duplicated().sum()
    })

quality_report = pd.DataFrame(report)


# =========================================================
# DISPLAY REPORT
# =========================================================

print(quality_report)


# =========================================================
# EXPORT REPORT
# =========================================================

quality_report.to_excel(
    'Data_Quality_Report.xlsx',
    index=False
)

print("\nData Quality Report Generated Successfully")

   Table_Name   Rows  Columns  Null_Values  Duplicate_Rows
0       Sales  74417       12            0              83
1    Products   1202        8            0               0
2   Inventory  50076        7            0               0
3     Returns   4617        7            0               0
4      Vendor  30085       11            0               0
5  Promotions    202        6            0               0
6   Customers  15036        4            0               0

Data Quality Report Generated Successfully


In [30]:
import pandas as pd
from prophet import Prophet


# ==========================================
# LOAD DATA
# ==========================================

sales = pd.read_excel(
    "Retail_Profit_Leakage_Dataset.xlsx",
    sheet_name='Sales_Transactions'
)


# ==========================================
# FIX DATE
# ==========================================

sales['Order_Date'] = pd.to_datetime(
    sales['Order_Date'],
    errors='coerce'
)

# Remove invalid dates
sales = sales.dropna(
    subset=['Order_Date']
)


# ==========================================
# DAILY SALES
# ==========================================

daily_sales = sales.groupby(
    'Order_Date'
)['Quantity'].sum().reset_index()


# Prophet format
daily_sales.columns = ['ds', 'y']


# ==========================================
# BUILD MODEL
# ==========================================

model = Prophet()

model.fit(daily_sales)


# ==========================================
# NEXT 4 WEEKS
# ==========================================

future = model.make_future_dataframe(
    periods=28
)


# ==========================================
# FORECAST
# ==========================================

forecast = model.predict(future)


# ==========================================
# FINAL TABLE
# ==========================================

forecast_table = forecast[[
    'ds',
    'yhat',
    'yhat_lower',
    'yhat_upper'
]].copy()


# Remove negative values
forecast_table['yhat'] = forecast_table[
    'yhat'
].clip(lower=0)

forecast_table['yhat_lower'] = forecast_table[
    'yhat_lower'
].clip(lower=0)

forecast_table['yhat_upper'] = forecast_table[
    'yhat_upper'
].clip(lower=0)


# ==========================================
# SHOW OUTPUT
# ==========================================

print(
    forecast_table.tail(28)
)


# ==========================================
# SAVE FILE
# ==========================================

forecast_table.to_excel(
    'Demand_Forecast.xlsx',
    index=False
)


print("\nForecast Generated Successfully")

21:19:21 - cmdstanpy - INFO - Chain [1] start processing
21:19:21 - cmdstanpy - INFO - Chain [1] done processing


            ds  yhat  yhat_lower  yhat_upper
215 2025-12-07   0.0         0.0   53.656347
216 2025-12-08   0.0         0.0   72.639641
217 2025-12-09   0.0         0.0   46.684073
218 2025-12-10   0.0         0.0    0.000000
219 2025-12-11   0.0         0.0   25.839511
220 2025-12-12   0.0         0.0   19.230539
221 2025-12-13   0.0         0.0    0.000000
222 2025-12-14   0.0         0.0    0.000000
223 2025-12-15   0.0         0.0   33.195813
224 2025-12-16   0.0         0.0    0.000000
225 2025-12-17   0.0         0.0    0.000000
226 2025-12-18   0.0         0.0    0.000000
227 2025-12-19   0.0         0.0    0.000000
228 2025-12-20   0.0         0.0    0.000000
229 2025-12-21   0.0         0.0    0.000000
230 2025-12-22   0.0         0.0    0.000000
231 2025-12-23   0.0         0.0    0.000000
232 2025-12-24   0.0         0.0    0.000000
233 2025-12-25   0.0         0.0    0.000000
234 2025-12-26   0.0         0.0    0.000000
235 2025-12-27   0.0         0.0    0.000000
236 2025-1

In [31]:
# CUSTOMER SEGMENTATION

# Total Spend
sales['Total_Spend'] = (
    sales['Selling_Price'] *
    sales['Quantity']
)

# Customer Metrics
customer = sales.groupby(
    'Customer_ID'
).agg({

    'Total_Spend': 'sum',

    'Order_ID': 'nunique'

}).reset_index()

# Rename Column
customer.rename(
    columns={
        'Order_ID': 'Order_Frequency'
    },
    inplace=True
)

# Avg Order Value
customer['Avg_Order_Value'] = (
    customer['Total_Spend'] /
    customer['Order_Frequency']
)

# Merge Returns with Sales
return_data = returns.merge(
    sales[['Order_ID', 'Customer_ID']],
    on='Order_ID',
    how='left'
)

# Return Frequency
return_freq = return_data.groupby(
    'Customer_ID'
).size().reset_index(
    name='Return_Frequency'
)

# Merge Return Data
customer = customer.merge(
    return_freq,
    on='Customer_ID',
    how='left'
)

# Fill Null Values
customer['Return_Frequency'] = customer[
    'Return_Frequency'
].fillna(0)

# Default Segment
customer['Segment'] = 'Regular Customers'

# High Value Loyal
customer.loc[
    (customer['Total_Spend'] > 50000) &
    (customer['Order_Frequency'] > 10),
    'Segment'
] = 'High Value Loyal'

# Deal Hunters
customer.loc[
    customer['Avg_Order_Value'] < 1000,
    'Segment'
] = 'Deal Hunters'

# One Time Buyers
customer.loc[
    customer['Order_Frequency'] == 1,
    'Segment'
] = 'One Time Buyers'

# High Return Risk
customer.loc[
    customer['Return_Frequency'] > 5,
    'Segment'
] = 'High Return Risk'

# Output
print(customer.head())

# Save
customer.to_excel(
    'Customer_Segmentation.xlsx',
    index=False
)

print("Customer Segmentation Completed")

  Customer_ID  Total_Spend  Order_Frequency  Avg_Order_Value  \
0  CUST000001      1840.58                4          460.145   
1  CUST000002      2144.49               10          214.449   
2  CUST000003      1365.46                5          273.092   
3  CUST000004      1123.44                3          374.480   
4  CUST000005       420.03                2          210.015   

   Return_Frequency       Segment  
0               0.0  Deal Hunters  
1               1.0  Deal Hunters  
2               1.0  Deal Hunters  
3               1.0  Deal Hunters  
4               1.0  Deal Hunters  
Customer Segmentation Completed


In [32]:
customer['Segment'].value_counts()

Segment
Deal Hunters         14473
One Time Buyers        400
Regular Customers       56
Name: count, dtype: int64

In [33]:
# AUTOMATED INSIGHTS REPORT

import pandas as pd

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet


# =========================================
# TOP 5 LOSS DRIVERS
# =========================================

top_loss = sales.groupby(
    'SKU_ID'
)['Discount_Amount'].sum().reset_index()

top_loss = top_loss.sort_values(
    by='Discount_Amount',
    ascending=False
).head(5)


# =========================================
# WORST 3 VENDORS
# =========================================

vendor['Delay_Days'] = (
    pd.to_datetime(
        vendor['Actual_Delivery_Date'],
        errors='coerce'
    ) -
    pd.to_datetime(
        vendor['Expected_Delivery_Date'],
        errors='coerce'
    )
).dt.days

worst_vendors = vendor.groupby(
    'Vendor_ID'
)['Delay_Days'].mean().reset_index()

worst_vendors = worst_vendors.sort_values(
    by='Delay_Days',
    ascending=False
).head(3)


# =========================================
# STOCKOUT HOTSPOTS
# =========================================

stockout = inventory[
    inventory['Closing_Stock'] == 0
]

stockout_hotspots = stockout.groupby(
    ['Store_ID', 'SKU_ID']
).size().reset_index(
    name='Stockout_Count'
)


# =========================================
# BEST 5 SKUS
# =========================================

best_skus = sales.groupby(
    'SKU_ID'
)['Quantity'].sum().reset_index()

best_skus = best_skus.sort_values(
    by='Quantity',
    ascending=False
).head(5)


# =========================================
# SAVE EXCEL REPORT
# =========================================

with pd.ExcelWriter(
    'Insights_Report.xlsx'
) as writer:

    top_loss.to_excel(
        writer,
        sheet_name='Loss_Drivers',
        index=False
    )

    worst_vendors.to_excel(
        writer,
        sheet_name='Worst_Vendors',
        index=False
    )

    stockout_hotspots.to_excel(
        writer,
        sheet_name='Stockouts',
        index=False
    )

    best_skus.to_excel(
        writer,
        sheet_name='Best_SKUs',
        index=False
    )

print("Excel Report Generated")


# =========================================
# PDF REPORT
# =========================================

doc = SimpleDocTemplate(
    "Insights_Report.pdf"
)

styles = getSampleStyleSheet()

content = []

# Title
content.append(
    Paragraph(
        "Automated Insights Report",
        styles['Title']
    )
)

content.append(Spacer(1, 20))


# Top 5 Loss Drivers
content.append(
    Paragraph(
        "Top 5 Loss Drivers",
        styles['Heading2']
    )
)

for i in range(len(top_loss)):
    content.append(
        Paragraph(
            f"{top_loss.iloc[i]['SKU_ID']} "
            f"- Loss: "
            f"{round(top_loss.iloc[i]['Discount_Amount'],2)}",
            styles['BodyText']
        )
    )


content.append(Spacer(1, 20))


# Worst 3 Vendors
content.append(
    Paragraph(
        "Worst 3 Vendors",
        styles['Heading2']
    )
)

for i in range(len(worst_vendors)):
    content.append(
        Paragraph(
            f"{worst_vendors.iloc[i]['Vendor_ID']} "
            f"- Delay: "
            f"{round(worst_vendors.iloc[i]['Delay_Days'],2)} days",
            styles['BodyText']
        )
    )


content.append(Spacer(1, 20))


# Stockout Hotspots
content.append(
    Paragraph(
        "Stockout Hotspots",
        styles['Heading2']
    )
)

for i in range(5):
    content.append(
        Paragraph(
            f"Store: "
            f"{stockout_hotspots.iloc[i]['Store_ID']} "
            f"- SKU: "
            f"{stockout_hotspots.iloc[i]['SKU_ID']}",
            styles['BodyText']
        )
    )


content.append(Spacer(1, 20))


# Best 5 SKUs
content.append(
    Paragraph(
        "Best 5 SKUs To Promote",
        styles['Heading2']
    )
)

for i in range(len(best_skus)):
    content.append(
        Paragraph(
            f"{best_skus.iloc[i]['SKU_ID']} "
            f"- Quantity Sold: "
            f"{best_skus.iloc[i]['Quantity']}",
            styles['BodyText']
        )
    )


# Save PDF
doc.build(content)

print("PDF Report Generated")

C:\Users\USER\AppData\Local\Temp\ipykernel_19712\3547528792.py:33: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  pd.to_datetime(
C:\Users\USER\AppData\Local\Temp\ipykernel_19712\3547528792.py:37: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  pd.to_datetime(


Excel Report Generated
PDF Report Generated


In [34]:
customer.head(5)

,Customer_ID,Total_Spend,Order_Frequency,Avg_Order_Value,Return_Frequency,Segment
0,CUST000001,1840.58,4,460.145,0.0,Deal Hunters
1,CUST000002,2144.49,10,214.449,1.0,Deal Hunters
2,CUST000003,1365.46,5,273.092,1.0,Deal Hunters
3,CUST000004,1123.44,3,374.480,1.0,Deal Hunters
4,CUST000005,420.03,2,210.015,1.0,Deal Hunters
